<a href="https://colab.research.google.com/github/garnfelfel/Deep_Learning_Homeworks/blob/main/TransformerEncoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Imports**


In [60]:
import os
import glob
import random
import shutil

In [61]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [62]:
source_data_dir =  '/content/drive/MyDrive/EEG-to-Text/splits'
train_dir = os.path.join(source_data_dir, 'train')
test_dir = os.path.join(source_data_dir, 'test' )
val_dir = os.path.join(source_data_dir, 'val')
!ls "$source_data_dir"
print("Train data")
!ls "$train_dir"
print("Test data")
!ls "$test_dir"
print("Validation data")
!ls "$val_dir"

test  train  val
Train data
 10_sub-02.nwb		   geen_sub-03.nwb	     sok_sub-10.nwb
 10_sub-05.nwb		   geen_sub-04.nwb	     spreuk_sub-01.nwb
 10_sub-07.nwb		   geen_sub-07.nwb	     spreuk_sub-02.nwb
 10_sub-08.nwb		   geen_sub-08.nwb	     spreuk_sub-04.nwb
 11_sub-02.nwb		   geen_sub-09.nwb	     spreuk_sub-05.nwb
 11_sub-03.nwb		   geen_sub-10.nwb	     spreuk_sub-10.nwb
 11_sub-05.nwb		   gefluit_sub-01.nwb	     sprong_sub-01.nwb
 11_sub-06.nwb		   gefluit_sub-04.nwb	     sprong_sub-02.nwb
 11_sub-08.nwb		   gefluit_sub-05.nwb	     sprong_sub-03.nwb
 11_sub-09.nwb		   gefluit_sub-06.nwb	     sprong_sub-05.nwb
 11_sub-10.nwb		   gefluit_sub-07.nwb	     sprong_sub-06.nwb
 12_sub-01.nwb		   groen_sub-01.nwb	     sprong_sub-07.nwb
 12_sub-02.nwb		   groen_sub-02.nwb	     sprong_sub-09.nwb
 12_sub-07.nwb		   groen_sub-03.nwb	    '`s_sub-01.nwb'
 12_sub-09.nwb		   groen_sub-04.nwb	    '`s_sub-03.nwb'
 12_sub-10.nwb		   groen_sub-05.nwb	    '`s_sub-05.nwb'
 1_sub-03.nwb		   groen_sub-06.nwb	 

# **Debugging splits**

In [63]:
# Define how many files for debugging
N_TRAIN = 100
N_VAL = 15
N_TEST = 15

# creating debug splits
small_splits_dir = '/content/drive/MyDrive/EEG-to-Text/small_splits'

small_train_dir = os.path.join(small_splits_dir, 'train')
small_val_dir = os.path.join(small_splits_dir, 'val')
small_test_dir = os.path.join(small_splits_dir, 'test')

!ls "$source_data_dir"
print("Train data")
!ls "$small_train_dir"
print("Test data")
!ls "$small_test_dir"
print("Validation data")
!ls "$small_val_dir"

test  train  val
Train data
12_sub-01.nwb		 groen_sub-04.nwb     onmiddellijk_sub-02.nwb
bevrijd_sub-05.nwb	 groen_sub-10.nwb     onmiddellijk_sub-08.nwb
binnenplaats_sub-05.nwb  helemaal_sub-04.nwb  schold_sub-01.nwb
braadde_sub-02.nwb	 het_sub-02.nwb       tak_sub-04.nwb
buurt_sub-06.nwb	 hierop_sub-06.nwb    teruggekregen_sub-06.nwb
daarna_sub-10.nwb	 hun_sub-07.nwb       teruggekregen_sub-09.nwb
die_sub-10.nwb		 komt_sub-09.nwb      uittrekken_sub-08.nwb
door_sub-06.nwb		 mooi_sub-09.nwb      verlost_sub-10.nwb
groen_sub-01.nwb	 naar_sub-10.nwb
groen_sub-03.nwb	 of_sub-04.nwb
Test data
aan_sub-01.nwb	direct_sub-02.nwb  kwamen_sub-02.nwb
als_sub-08.nwb	had_sub-04.nwb
Validation data
hem_sub-02.nwb	 nachtegalen_sub-02.nwb  wak_sub-06.nwb
naar_sub-01.nwb  spreuk_sub-09.nwb


# **Step 1**

Create vocab dict

In [64]:
import torch

# Create a vocabulary
vocab = "abcdefghijklmnopqrstuvwxyz0123456789'"
char_to_int = {char: i for i, char in enumerate(vocab)}
int_to_char = {i: char for i, char in enumerate(vocab)}

# Add the CTC "blank" token
BLANK_TOKEN_INDEX = len(vocab)
char_to_int['<BLANK>'] = BLANK_TOKEN_INDEX
int_to_char[BLANK_TOKEN_INDEX] = '<BLANK>'

# The total number of classes
NUM_GRAPHEMES = len(char_to_int)
print(f"Total grapheme classes (including blank): {NUM_GRAPHEMES}")

Total grapheme classes (including blank): 38


# **Step 2**
Create the PyTorch Dataset (Mini Dataset for debugging purpose)
This is the Dataset class that only loads the pre-processed .pt files.

In [65]:
from torch.utils.data import Dataset, DataLoader
import os

class EEGWordDataset(Dataset):
    """
    A Dataset that loads *pre-processed* tensors from Drive.
    """
    def __init__(self, data_dir, char_map):
        """
        Args:
            data_dir (str): Path to the folder (".../processed_frames/train/").
            char_map (dict): The 'char_to_int' mapping.
        """
        self.data_dir = data_dir
        self.char_map = char_map

        self.samples = []
        for f in os.listdir(data_dir):
            if f.endswith('.pt'):
                # Extract word from "word_subject.pt"
                word = f.split('_')[0]
                self.samples.append((os.path.join(data_dir, f), word))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, word = self.samples[idx]

        # Load the *already processed* tensor
        x_tensor = torch.load(file_path)

        # Encode the target word
        target = [self.char_map[char] for char in word.lower() if char in self.char_map]
        y_tensor = torch.tensor(target, dtype=torch.long)

        return x_tensor, y_tensor

In [66]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import os

class EEGWordDataset_MultiSubject(Dataset):
    """
    Loads data from multiple subjects by padding the channel dimension.
    """
    def __init__(self, data_dir, char_map, max_channels):
        self.data_dir = data_dir
        self.char_map = char_map
        self.max_channels = max_channels # Max channels found in your dataset

        self.samples = []
        for f in os.listdir(data_dir):
            if f.endswith('.pt'):
                word = f.split('_')[0]
                self.samples.append((os.path.join(self.data_dir, f), word))

        # print(f"Loaded {len(self.samples)} samples from {data_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, word = self.samples[idx]

        # Load the 3D tensor
        # Shape: [num_frames, window_size, num_channels]
        x_tensor = torch.load(file_path)

        num_frames, window_size, num_channels = x_tensor.shape

        # Pad the channel dimension if necessary
        if num_channels < self.max_channels:
            # Calculate padding needed
            pad_width = self.max_channels - num_channels

            # Pad LAST dimension (channels)
            # (pad_left, pad_right, pad_top, pad_bottom, pad_front, pad_back)
            # We only want to pad the last dim: (0, pad_width)
            x_tensor = F.pad(x_tensor, (0, pad_width), "constant", 0)

        # Flatten features
        # Shape: [num_frames, window_size * self.max_channels]
        x_tensor = x_tensor.flatten(start_dim=1)

        # Encode the target word
        target = [self.char_map[char] for char in word.lower() if char in self.char_map]
        y_tensor = torch.tensor(target, dtype=torch.long)

        return x_tensor, y_tensor

# Step 3
the Collate Function :

This function takes a list of samples from the Dataset (which all have different lengths) and bundles them into a single, padded batch.

In [67]:
import torch.nn.utils.rnn as rnn_utils

def ctc_collate_fn(batch):
    """
    Processes a list of (sequence, target) tuples
    and turns them into padded batches.
    """
    # Separate sequences and targets
    sequences = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    # print(sequences)
    # print(targets)
    # Get original lengths (for CTC loss)
    seq_lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)

    # Pad the sequences (batch_first=False for model)
    # Output shape: [SeqLen, BatchSize, FeatureDim]
    padded_seqs = rnn_utils.pad_sequence(
        sequences,
        batch_first=False,
        padding_value=0.0
    )

    # Pad the targets (batch_first=True for loss)
    # Output shape: [BatchSize, MaxTargetLen]
    padded_targets = rnn_utils.pad_sequence(
        targets,
        batch_first=True,
        padding_value=0
    )

    return padded_seqs, padded_targets, seq_lengths, target_lengths


# **Step 4** : CTC-Transformer Model

This code defines a model that uses a **TransformerEncoder** to process the HFA sequence and a final linear layer to project the output into the grapheme space, suitable for the CTC loss function.

In [68]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    """
    Standard positional encoding for Transformer models.
    This adds temporal context to the HFA features.
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor, shape [seq_len, batch_size, embedding_dim]
        """
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

class BCIToTextModel(nn.Module):
    """
    CTC-Transformer model for BCI-to-Text decoding based on.

    This model implements the "Core Architecture":
    1.  An input layer to project HFA features to the model's dimension.
    2.  Positional encoding to inject sequence information.
    3.  A Multi-Head Transformer Encoder to capture dependencies.
    4.  An output layer to map encoder hidden states to grapheme probabilities.
    """
    def __init__(self,
                 input_feat_dim: int,
                 num_graphemes: int,
                 nhead: int = 8,
                 d_model: int = 512,
                 num_encoder_layers: int = 6,
                 dim_feedforward: int = 2048,
                 dropout: float = 0.1):
        """
        Args:
            input_feat_dim (int): Dimension of the input HFA feature vector.
                                  (Derived from 1103 electrodes + windowing)
            num_graphemes (int): The number of output classes (letter, special chars + CTC blank token).
            nhead (int): Number of heads in the "Multi-Head Transformer Encoder".
            d_model (int): The hidden dimension of the transformer.
            num_encoder_layers (int): Number of layers in the encoder.
            dim_feedforward (int): Dimension of the FFN inside the transformer.
            dropout (float): Dropout rate.
        """
        super().__init__()
        self.d_model = d_model

        # Input feature projection : Mapping the high-dimensional HFA vector to the model's working dimension
        self.input_projection = nn.Linear(input_feat_dim, d_model)

        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # CTC-Transformer (Multi-Head Transformer Encoder)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=False
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_encoder_layers
        )

        # Output Layer
        # Projects encoder output to the grapheme vocabulary size
        self.fc_out = nn.Linear(d_model, num_graphemes)

        # LogSoftmax for CTC Loss
        # nn.CTCLoss expects log-probabilities as input
        self.log_softmax = nn.LogSoftmax(dim=2)

    def forward(self, src: torch.Tensor, src_key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            src (torch.Tensor): The input XHFA sequence.
                                Shape: [seq_len, batch_size, input_feat_dim]
            src_key_padding_mask (torch.Tensor, optional):
                                Mask for padded elements in the batch.
                                Shape: [batch_size, seq_len]

        Returns:
            torch.Tensor: Log-probabilities for CTC Loss.
                          Shape: [seq_len, batch_size, num_graphemes]
        """
        # Project features and add positional encoding
        src = self.input_projection(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)

        # Pass through the Transformer Encoder
        # src_key_padding_mask informs the model which parts of the sequence are padding
        memory = self.transformer_encoder(
            src,
            src_key_padding_mask=src_key_padding_mask
        )

        # Project to grapheme vocabulary
        output = self.fc_out(memory)

        # Apply LogSoftmax for nn.CTCLoss
        return self.log_softmax(output)

# **Step 5: Set Up and Run the Training Loop**

In [69]:
import torch.optim as optim
# Define Paths and Parameters ---
PROCESSED_DIR = '/content/drive/MyDrive/EEG-to-Text/small_preprocessed_frames'
TRAIN_DIR = os.path.join(PROCESSED_DIR, 'train')
VAL_DIR = os.path.join(PROCESSED_DIR, 'val')
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_CHANNELS = 127
NUM_EPOCHS = 10

print(f"Using device: {DEVICE}")

# Instantiate Datasets
train_dataset = EEGWordDataset_MultiSubject(
    data_dir=TRAIN_DIR,
    char_map=char_to_int,
    max_channels=MAX_CHANNELS
)
val_dataset = EEGWordDataset_MultiSubject(
    data_dir=VAL_DIR,
    char_map=char_to_int,
    max_channels=MAX_CHANNELS
)


# Instantiate DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ctc_collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=4
)

# Get Feature Dimension ---
# Load one sample to find the feature dimension
sample_seq, _ = train_dataset[0]
HFA_FEATURE_DIM = sample_seq.shape[1] # [SeqLen, FeatureDim] -> get FeatureDim
# print(f"Detected Feature Dimension: {HFA_FEATURE_DIM}")

# Instantiate Model, Loss, and Optimizer ---
model = BCIToTextModel(
    input_feat_dim=HFA_FEATURE_DIM,
    num_graphemes=NUM_GRAPHEMES,
    nhead=8,
    d_model=512,
    num_encoder_layers=6,
    dim_feedforward=2048
).to(DEVICE)

# CTCLoss expects the 'blank' token index
ctc_loss = nn.CTCLoss(blank=BLANK_TOKEN_INDEX)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# The Training Loop ---

model.train() # Set model to training mode

for epoch in range(NUM_EPOCHS):
    # print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    epoch_loss = 0

    for i, batch in enumerate(train_loader):
        # Move data to the GPU
        padded_seqs, padded_targets, seq_lengths, target_lengths = [
            d.to(DEVICE) for d in batch
        ]

        # --- Create padding mask for the Transformer Encoder ---
        # This tells the Transformer to ignore padded time steps
        # Shape: [BatchSize, SeqLen]
        max_seq_len = padded_seqs.shape[0]
        padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                        .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))

        # --- Forward pass ---
        optimizer.zero_grad()
        log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)

        # Calculate Loss ---
        # CTCLoss requires log_probs in [SeqLen, BatchSize, Classes]
        loss = ctc_loss(
            log_probs,
            padded_targets,
            seq_lengths,
            target_lengths
        )

        # --- Backward pass ---
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        if (i + 1) % 10 == 0:
            print(f"  Batch {i+1}/{len(train_loader)}, Loss: {loss.item():.4f}")


    print(f"End of Epoch {epoch+1}, Average Loss: {epoch_loss / len(train_loader):.4f}")

print("\n--- Training Complete ---")

Using device: cpu


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


End of Epoch 1, Average Loss: 113.4184
End of Epoch 2, Average Loss: 9.1016
End of Epoch 3, Average Loss: 4.3331
End of Epoch 4, Average Loss: 4.9513
End of Epoch 5, Average Loss: 5.1918
End of Epoch 6, Average Loss: 5.1210
End of Epoch 7, Average Loss: 4.8995
End of Epoch 8, Average Loss: 4.4958
End of Epoch 9, Average Loss: 4.1003
End of Epoch 10, Average Loss: 3.7990

--- Training Complete ---


# **Linguistic Refinement**

In [70]:
!pip install pyctcdecode jiwer huggingface_hub
!pip install https://github.com/kpu/kenlm/archive/master.zip

  Using cached https://github.com/kpu/kenlm/archive/master.zip
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [71]:
# Imports
import torch
import torch.nn as nn
import jiwer
import json
import time
import os
from huggingface_hub import hf_hub_download
from pyctcdecode import build_ctcdecoder

In [72]:
print("/nPreparing Decoder ---")

# --- 1a. Download the Language Model ---
repo_id = "BramVanroy/kenlm_wikipedia_nl"
filename = "wiki_nl_token.arpa.bin" # The faster binary file
print(f"Downloading {filename} from {repo_id}...")

try:
    LANGUAGE_MODEL_PATH = hf_hub_download(
        repo_id=repo_id,
        filename=filename
    )
    print(f"Language Model saved to: {LANGUAGE_MODEL_PATH}")
except Exception as e:
    print(f"Error downloading LM: {e}")
    print("Please check your internet connection or the Hugging Face repo.")

/nPreparing Decoder ---
Language Model saved to: /root/.cache/huggingface/hub/models--BramVanroy--kenlm_wikipedia_nl/snapshots/0b9a2ce5bbec0b16486808182b79c9d0dc2a5e28/wiki_nl_token.arpa.bin


In [73]:
# Create the Vocabulary JSON File
LABELS_FILE_PATH = '/content/drive/MyDrive/EEG-to-Text/labels.json'
labels = [int_to_char.get(i, '') for i in range(len(int_to_char))]
labels[BLANK_TOKEN_INDEX] = ""  # Blank token must be an empty string
with open(LABELS_FILE_PATH, 'w') as f:
    json.dump(labels, f)
print(f"Created {LABELS_FILE_PATH} with {len(labels)} graphemes.")


Created /content/drive/MyDrive/EEG-to-Text/labels.json with 38 graphemes.


In [74]:
# Load the CTC Beam Search Decoder
try:
    # Use build_ctcdecoder() instead of BeamSearchDecoder()
    decoder = build_ctcdecoder(
        labels,
        kenlm_model_path=LANGUAGE_MODEL_PATH,
        alpha=0.5,  # How much to trust the LM
        beta=1.0,   # How much to reward word length
    )
    print("Successfully loaded CTC Beam Search Decoder.")
except Exception as e:
    print(f"Error loading Decoder: {e}")

Successfully loaded CTC Beam Search Decoder.


In [75]:
# Helper Functions ---
def decode_target_batch(target_ids, int_to_char):
    """Converts target indices back to strings."""
    decoded_batch = []
    for i in range(target_ids.shape[0]):
        seq = target_ids[i]
        decoded_word = [int_to_char.get(idx.item(), '?') for idx in seq if idx != 0] # 0 is pad
        decoded_batch.append("".join(decoded_word))
    return decoded_batch

def greedy_decode_batch(log_probs, int_to_char, blank_index):
    """Decodes a batch of log_probs greedily (NO LM)."""
    decoded_batch = []
    best_path = torch.argmax(log_probs, dim=2)
    for i in range(best_path.shape[1]):
        seq = best_path[:, i]
        decoded_word = []
        last_char_idx = -1
        for char_idx in seq:
            if char_idx.item() == last_char_idx or char_idx.item() == blank_index:
                last_char_idx = char_idx.item()
                continue
            decoded_word.append(int_to_char.get(char_idx.item(), '?'))
            last_char_idx = char_idx.item()
        decoded_batch.append("".join(decoded_word))
    return decoded_batch

def beam_search_decode_batch(log_probs, decoder):
    """Decodes a batch using the LM-Refined Beam Search."""
    log_probs_permuted = log_probs.permute(1, 0, 2)
    probs = torch.exp(log_probs_permuted).cpu().numpy()

    decoded_batch = []
    for i in range(probs.shape[0]):
        # This is the "Linguistic Refinement" step
        text = decoder.decode(probs[i])
        decoded_batch.append(text)
    return decoded_batch

In [76]:

# Run Evaluation Loop
print("\nStarting Model Evaluation ---")
model.eval()
model.to(DEVICE)

all_targets = []
all_greedy_preds = []
all_lm_preds = []

start_time = time.time()
with torch.no_grad():
    for i, batch in enumerate(val_loader):
        padded_seqs, padded_targets, seq_lengths, target_lengths = [
            d.to(DEVICE) for d in batch
        ]

        max_seq_len = padded_seqs.shape[0]
        padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                        .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))

        # Get model output (log-probabilities)
        log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)

        # Decode Ground Truth
        truth_text = decode_target_batch(padded_targets.cpu(), int_to_char)

        # Decode Greedy (No Refinement)
        greedy_text = greedy_decode_batch(log_probs.cpu(), int_to_char, BLANK_TOKEN_INDEX)

        #  Decode with Linguistic Refinement
        lm_text = beam_search_decode_batch(log_probs, decoder)

        all_targets.extend(truth_text)
        all_greedy_preds.extend(greedy_text)
        all_lm_preds.extend(lm_text)

        if (i + 1) % 10 == 0:
            print(f"  Processed batch {i+1} / {len(val_loader)}")

end_time = time.time()
print(f"Evaluation Complete ({end_time - start_time:.2f}s)")

#Calculate and Print Final Metrics
greedy_wer = jiwer.wer(all_targets, all_greedy_preds)
lm_wer = jiwer.wer(all_targets, all_lm_preds)

print("\nPerformance Results ")
print(f"  Greedy Decoding (No LM) WER:     {greedy_wer * 100:.2f}%")
print(f"  Linguistic Refinement (LM) WER:  {lm_wer * 100:.2f}%")
print("-" * 30)
improvement = greedy_wer - lm_wer
print(f"  Improvement from LM:             {improvement * 100:.2f}%")

# Show Example Decodings
print("\nExample Decodings")
print("TARGET".ljust(15), "| GREEDY".ljust(15), "| LM REFINED")
print("-" * 45)
for i in range(min(20, len(all_targets))): # Show first 20 examples
    print(f"{all_targets[i]: <15} | {all_greedy_preds[i]: <15} | {all_lm_preds[i]}")


Starting Model Evaluation ---
Evaluation Complete (2.77s)

Performance Results 
  Greedy Decoding (No LM) WER:     100.00%
  Linguistic Refinement (LM) WER:  100.00%
------------------------------
  Improvement from LM:             0.00%

Example Decodings
TARGET          | GREEDY        | LM REFINED
---------------------------------------------
hem             |                 | 
spreuk          |                 | 
nr              |                 | 
nchteglen       |                 | 
wk              |                 | 
